# 07 — Qwen2.5-7B-Instruct (LLM locale via ollama)

Primo dei tre notebook (07–09) che valutano **LLM generalisti eseguiti in locale** sullo stesso
campione condiviso dei metodi 01–04 e 06. A differenza di BART/PEGASUS/PRIMERA — modelli
*specializzati* in summarization, qui fine-tuned o addestrati su corpora di news — un LLM
instruction-tuned come **Qwen2.5-7B-Instruct** riceve semplicemente un prompt zero-shot che gli
chiede il riassunto.

Scelte che replicano la corsa originale (vedi `notebooks/llm/Summarization_LLM_Evaluation.docx`):
prompt di sistema e utente in inglese, `temperature=0.3`, e — **solo per questo modello** —
`max_tokens=200` (gli altri due usano 300): i suoi riassunti sono quindi strutturalmente più
corti, da tenere presente leggendo precisione/recall.

Il prefisso `/no_think` nel prompt è un residuo della corsa originale (pensato per i modelli
Qwen3 con reasoning); per Qwen2.5 è inerte e lo conserviamo per fedeltà.

## Provenienza dei risultati committati e ripresa

I file committati (`results/summaries/qwen_sample.tsv` e le metriche) provengono dalla
**corsa di questo notebook via ollama** (100/100 esempi, 2026-07-16), che ha rigenerato da
zero i riassunti sostituendo quelli importati dalla corsa originale di Federica via LM Studio
(Mac M4, 2026-07-16; i CSV originali restano archiviati in `notebooks/llm/`). Rispetto alla
corsa originale ci sono due differenze deliberate:

- il documento passa da `su.prepara_documento` (separatore `` ||||| `` → newline), mentre la
  corsa originale inviava il testo grezzo con il separatore incluso;
- niente `extra_body={"enable_thinking": False}` (parametro specifico di LM Studio; ollama
  lo ignorerebbe).

⚠️ **Attenzione alla ripresa**: il ciclo condiviso salta i `row_id` già presenti nel TSV, quindi
rieseguire la generazione su un TSV esistente **aggiunge solo le righe mancanti** — utile per
completare una corsa ollama interrotta, ma da evitare su un file prodotto da un backend o con
una configurazione diversi (es. un re-import della corsa LM Studio), perché mescolerebbe le due
corse nello stesso file. In quel caso **eliminare prima** `results/summaries/qwen_sample.tsv`
e rigenerare tutto (rieseguendo poi la valutazione).

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [1]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'qwen'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
MODELLO    = 'qwen2.5:7b-instruct'  # tag ollama; verificare il tag esatto con `ollama list`
OLLAMA_URL = 'http://localhost:11434/v1'   # endpoint OpenAI-compatibile di ollama

MAX_TOKENS  = 200         # come la corsa LM Studio originale
TEMPERATURE = 0.3
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = ('/no_think\n\n'
                 'Summarize the following document into a comprehensive summary: {documento}')
ETICHETTA   = 'Qwen '
NOTE_CONFIG = ('prompt zero-shot in inglese; max_tokens=200 come la corsa originale '
               '(unico modello con limite 200 invece di 300)')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

print(f'Modello : {MODELLO} via {OLLAMA_URL}')
print(f'Output  : {OUT_PATH}')

Modello : qwen2.5:7b-instruct via http://localhost:11434/v1
Output  : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\qwen_sample.tsv


## Generazione dei riassunti

Il modello gira **fuori dal notebook**, nel server locale di ollama: qui usiamo solo il client
`openai` puntato all'endpoint OpenAI-compatibile (`http://localhost:11434/v1`). Prerequisiti:

```bash
ollama pull qwen2.5:7b-instruct
ollama serve   # se il servizio non e' gia' attivo
```

Il ciclo e la scrittura incrementale (con ripresa) sono quelli condivisi di `summ_utils`; una
risposta vuota del modello solleva un'eccezione, così la riga viene registrata come errore e
**non** scritta nel TSV (ritentabile alla corsa successiva).

In [2]:
from openai import OpenAI

client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')  # la chiave e' ignorata da ollama

def genera(documento):
    risposta = client.chat.completions.create(
        model=MODELLO,
        messages=[{'role': 'system', 'content': PROMPT_SYSTEM},
                  {'role': 'user', 'content': PROMPT_USER.format(documento=documento)}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto.strip()

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA)
scrittore.chiudi()

Qwen [1] media 26.6 s/esempio (saltati 0 gia' fatti)
Qwen [2] media 15.4 s/esempio (saltati 0 gia' fatti)
Qwen [3] media 11.9 s/esempio (saltati 0 gia' fatti)
Qwen [10] media 7.2 s/esempio (saltati 0 gia' fatti)
Qwen [20] media 6.7 s/esempio (saltati 0 gia' fatti)
Qwen [30] media 6.1 s/esempio (saltati 0 gia' fatti)
Qwen [40] media 5.6 s/esempio (saltati 0 gia' fatti)
Qwen [50] media 5.5 s/esempio (saltati 0 gia' fatti)
Qwen [60] media 5.4 s/esempio (saltati 0 gia' fatti)
Qwen [70] media 5.2 s/esempio (saltati 0 gia' fatti)
Qwen [80] media 5.2 s/esempio (saltati 0 gia' fatti)
Qwen [90] media 5.1 s/esempio (saltati 0 gia' fatti)
Qwen [100] media 5.1 s/esempio (saltati 0 gia' fatti)
Qwen Completato: 100 nuovi, 0 saltati, 0 errori, 512 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/qwen_sample_per_example.csv` e `…_aggregate.json`.

⚠️ La valutazione misura il TSV salvato **qualunque ne sia l'origine**, mentre il JSON aggregato
riporta la configurazione dichiarata qui sotto: oggi le due cose coincidono (il TSV committato
è la corsa ollama di questo notebook), ma se il TSV venisse re-importato da un'altra corsa
(es. LM Studio via `scripts/import_llm_results.py`, che scrive la propria provenienza nel JSON),
rieseguire questa cella sovrascriverebbe quella provenienza. Rieseguirla solo su un TSV
generato con la configurazione qui sopra.

In [3]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'modello': MODELLO,
          'backend': 'ollama (endpoint OpenAI-compatibile)',
          'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': NOTE_CONFIG}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\qwen_sample_per_example.csv (100 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\qwen_sample_aggregate.json
{
  "rouge1_f1": 0.34466406372295516,
  "rouge1_p": 0.4083070595183305,
  "rouge1_r": 0.3079489725224917,
  "rouge2_f1": 0.09813008742492112,
  "rouge2_p": 0.12364488095650582,
  "rouge2_r": 0.08533858211490579,
  "rougeL_f1": 0.17301789351818372,
  "rougeL_p": 0.22191811171248463,
  "rougeL_r": 0.1482971598150461,
  "bleu": 0.056458790407167704,
  "meteor": 0.3172267653767472,
  "parole_generate": 143.77
}

Medie per split:
  test  (n=8): ROUGE-1 F1 = 0.324
  train (n=81): ROUGE-1 F1 = 0.342
  val   (n=11): ROUGE-1 F1 = 0.381


## Ispezione qualitativa

In [4]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : A Republican state Senate candidate in Connecticut, Ed Charamut, sent out a campaign mailer depicting his Democratic opponent, Jewish State Representative Matthew Lesser, holding money with exaggerated, beady eyes. The mailer was mailed to homes in Middletown, where Lesser is running for office. This imagery has been widely condemned as anti-Semitic and offensive.

Jewish groups and political opponents criticized the ad, describing it as an "age-old anti-Semitic trope." Republican Chair JB Roman
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 